# W3 · C2 与真实参数浏览器验证
复用已板测 W2 bit/HWH、原 DMA/DDR 和 W1 网络服务。无 HDMI。
上传本 Notebook 和 browser_w3_c2_r1.zip 到新 retina_browser_w3_r1 文件夹。
只在上一步结束且无报错后运行下一框；不要 Run All。FAULT 时保留内核和缓冲，不重复加载。


## 1. 验证下载包并展开
应打印 PACKAGE_BYTES_PASS。只读验证通过后写入新的子文件夹；尚未访问 FPGA。

In [ ]:
from pathlib import Path
import hashlib, json, zipfile
if 'browser_package_root' in globals():
    raise RuntimeError('Already extracted; continue to the next cell')
bundle_path = Path('browser_w3_c2_r1.zip')
expected_bundle_sha256 = 'fa0b51356e09e7912b88fd7e60662806b8f5bfa12028c700d517164a8dc111ce'
if hashlib.sha256(bundle_path.read_bytes()).hexdigest() != expected_bundle_sha256:
    raise ValueError('Wrong download bytes; stop before importing or loading FPGA')
with zipfile.ZipFile(bundle_path) as archive:
    members = archive.infolist()
    names = [member.filename for member in members]
    manifest = json.loads(archive.read('browser_manifest.json'))
    if len(names) != len(set(names)) or set(names) != set(manifest['artifacts']) | {'browser_manifest.json'}:
        raise ValueError('Unexpected archive inventory')
    if len(members) > 100 or sum(member.file_size for member in members) > 128 * 1024 * 1024:
        raise ValueError('Unexpected package size')
    for member in members:
        path = Path(member.filename)
        if path.is_absolute() or '..' in path.parts or '\\' in member.filename or ':' in member.filename or member.is_dir():
            raise ValueError('Unsafe package member')
        if member.filename != 'browser_manifest.json':
            data = archive.read(member)
            if {'bytes': len(data), 'sha256': hashlib.sha256(data).hexdigest()} != manifest['artifacts'][member.filename]:
                raise ValueError('Package mismatch: ' + member.filename)
    browser_package_root = (Path.cwd() / 'browser_w3_c2_r1').resolve()
    browser_package_root.mkdir(exist_ok=False)
    for member in members:
        destination = browser_package_root / member.filename
        if browser_package_root not in destination.resolve().parents:
            raise ValueError('Package path escaped directory')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
print('PACKAGE_BYTES_PASS: paired W2 and browser application bytes checked; FPGA untouched')


## 2. 检查环境并保留端口
应打印 BROWSER_PRECHECK_PASS 和实际版本。此步不加载FPGA。8090若被占用，保留报错，勿继续。

In [ ]:
import sys
if 'browser_context' in globals():
    raise RuntimeError('Already prepared; continue to the next cell')
sys.path.insert(0, str(browser_package_root))
from pynq_w3.launch import prepare, start, export_session, finish
browser_context = prepare(browser_package_root, port=8090)


## 3. 加载一次并启动服务
应打印 BROWSER_SERVICE_READY。尚未进行图像收发。出现红色报错则保留内核和上下文，不重复加载。

In [ ]:
browser_start_status = await start(browser_context)
print(browser_start_status)


## 4. 在 Windows 浏览器运行新任务
打开 http://192.168.2.99:8090/ ，若板卡 IP 变化使用第3框地址。每次等待任务完成且硬件实际参数与选择一致，再执行下一行。

|次序|图像|尺寸|算法|阈值|PL处理模式|
|---|---|---|---|---|---|
|1|im0001|512|A|28|最终掩膜 3|
|2|im0001|512|C2|28|最终掩膜 3|
|3|im0001|512|C2|40|最终掩膜 3|
|4|im0001|512|A|28|最终掩膜 3|
|5|im0002|512|C2|28|最终掩膜 3|
|6|im0001|256|C2|28|最终掩膜 3|
|7|im0001|256|C2|28|初始阈值候选 1|
|8|im0001|256|C2|28|输入旁路 0|

旁路只有原图，没有分割分数。选择改变后旧图变暗，点击“运行 FPGA”才发送真实新任务。
再选 im0001 /512/C2/28/最终掩膜，点击“连续运行20帧”，保持网页在前台，完成后截图。
PL核心窗口来自硬件周期且含流式等待；DMA事务含PL重叠；完整推理含配置、分配、DMA和CPU快照；浏览器延迟来自客户端单调时钟。纯搬运仍待独立实验。
本次28帧是交互集成验证，不是正式性能/完整数据集质量或稳定性验收。完成后返回Notebook运行下面状态框，应为IDLE、完成至少28帧，无故障/保留缓冲。


In [ ]:
print(browser_context['manager'].status())


## 5. 打包本次真实记录
须无在途/待执行任务。应打印 BROWSER_RECORDS_READY。完整会话包含每任务原packed NPY、PNG、JSON、CSV及浏览器绘制观测；尚待项目独立复核，不自动宣称验收通过。

In [ ]:
import shutil
browser_export_path = await export_session(browser_context)
shutil.copyfile(browser_export_path, Path.cwd() / 'browser_w3_session_bundle.zip')
print('DOWNLOAD: browser_w3_session_bundle.zip in this notebook folder')


## 6. 安全结束并下载
仅READY/IDLE且无在途或故障缓冲时关闭HTTP服务。FAULT时本框拒绝关闭，保留内核和原记录并反馈。
完成后Ctrl+S；File → Download as → Notebook下载执行记录。回文件列表下载 `browser_w3_session_bundle.zip`。反馈这两个文件和浏览器截图。

In [ ]:
await finish(browser_context)
